# 🔌 Smoke test: środowisko warsztatowe (Vertex AI + GitHub)

**Cel:** sprawdzić — na *firmowym* środowisku GCP — czy techniczne fundamenty warsztatu zadziałają:

1. **OAuth do Vertex AI** — połączenie z modelem Gemini przez `Authorization: Bearer`.
2. **Git clone** repozytorium warsztatowego z GitHub.

> ℹ️ Firmowy Vertex AI **nie udostępnia kluczy API** — autoryzacja idzie wyłącznie przez OAuth
> (konto użytkownika / service account / ADC). Tryb „API key" został usunięty jako nieistotny.

Sonda jest minimalna i używa surowego `requests` — pokazuje dokładny **status HTTP** i **treść błędu**,
co ułatwia diagnozę w korporacyjnym GCP (uprawnienia, włączone API, region, VPC-SC).

---
## Co trzeba mieć przed startem

1. **PROJECT_ID** firmowego projektu GCP.
2. Włączone **Vertex AI API** w tym projekcie (`aiplatform.googleapis.com`).
3. Konto / SA z rolą **`roles/aiplatform.user`**.
4. Potwierdzona dostępność modelu `gemini-2.5-flash-lite` w regionie **`europe-west4`**.
5. Dostęp sieciowy z Colab/VM do GitHub (dla testu clone).

## ⚙️ 1. Konfiguracja — UZUPEŁNIJ

In [ ]:
# ============================================================
# UZUPEŁNIJ wartości dla firmowego środowiska GCP
# ============================================================
PROJECT_ID = ""                       # TODO: ID firmowego projektu GCP
LOCATION   = "europe-west4"           # region warsztatu
MODEL      = "google/gemini-2.5-flash-lite"  # OpenAI-compatible nazwa modelu

# Repozytorium warsztatowe (test git clone)
REPO_URL = "https://github.com/JSerek/techland-genai-workshop.git"

# OpenAI-compatible endpoint Vertex AI (chat/completions)
BASE_URL = (
    f"https://{LOCATION}-aiplatform.googleapis.com/v1beta1"
    f"/projects/{PROJECT_ID}/locations/{LOCATION}/endpoints/openapi/chat/completions"
)

assert PROJECT_ID, "❌ Uzupełnij PROJECT_ID"
print("Endpoint:", BASE_URL)
print("Model:   ", MODEL)
print("Repo:    ", REPO_URL)

## 🔑 2. OAuth do Vertex AI (`Authorization: Bearer`)

Pozyskujemy token dostępu z **Application Default Credentials**.

- **W Google Colab:** poniższa komórka uruchomi logowanie Google (`auth.authenticate_user()`) — zaloguj się kontem mającym dostęp do firmowego projektu.
- **Lokalnie / na VM:** zadziała `gcloud auth application-default login` wykonane wcześniej, albo service account z `GOOGLE_APPLICATION_CREDENTIALS`.

In [ ]:
import json
import requests

token = None
try:
    # W Colab: zaloguj użytkownika (no-op poza Colab)
    try:
        from google.colab import auth as colab_auth
        colab_auth.authenticate_user()
        print("🔑 Zalogowano przez google.colab.auth")
    except ImportError:
        print("💻 Poza Colab — używam Application Default Credentials")

    import google.auth
    import google.auth.transport.requests

    creds, detected_project = google.auth.default(
        scopes=["https://www.googleapis.com/auth/cloud-platform"]
    )
    creds.refresh(google.auth.transport.requests.Request())
    token = creds.token
    print(f"✅ Token pozyskany. Wykryty projekt ADC: {detected_project}")
except Exception as e:
    print(f"❌ Nie udało się pozyskać tokenu OAuth: {type(e).__name__}: {e}")

## 🧪 3. Test zapytania do Gemini

In [ ]:
PAYLOAD = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "Jesteś asystentem testowym. Odpowiadaj krótko."},
        {"role": "user", "content": "Odpowiedz dokładnie jednym słowem: OK"},
    ],
    "temperature": 0.0,
}

result_oauth = False
if not token:
    print("⏭️  Brak tokenu — najpierw napraw komórkę OAuth powyżej.")
else:
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}",
    }
    print("Endpoint:", BASE_URL)
    try:
        resp = requests.post(BASE_URL, headers=headers, json=PAYLOAD, timeout=30)
        print(f"HTTP {resp.status_code}")
        if resp.status_code == 200:
            content = resp.json()["choices"][0]["message"]["content"]
            print(f"✅ SUKCES — odpowiedź modelu: {content!r}")
            result_oauth = True
        else:
            # Treść błędu jest najważniejsza do diagnozy (uprawnienia/API/region)
            print("❌ BŁĄD — treść odpowiedzi:")
            try:
                print(json.dumps(resp.json(), indent=2, ensure_ascii=False)[:2000])
            except Exception:
                print(resp.text[:2000])
    except Exception as e:
        print(f"❌ Wyjątek sieciowy: {type(e).__name__}: {e}")

## 📥 4. Test git clone repozytorium warsztatowego

Sprawdza, czy z firmowego środowiska da się sklonować repo (dostęp do GitHub + repo publiczne).
Klonujemy do katalogu tymczasowego i od razu sprzątamy.

In [ ]:
import subprocess
import tempfile
import shutil
import os

result_clone = False
tmp_dir = tempfile.mkdtemp(prefix="smoke_clone_")
clone_path = os.path.join(tmp_dir, "repo")
try:
    print(f"📥 Klonuję {REPO_URL} ...")
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, clone_path],
        capture_output=True, text=True, timeout=120,
    )
    if proc.returncode == 0:
        files = os.listdir(clone_path)
        print(f"✅ SUKCES — sklonowano. Zawartość (top-level): {sorted(files)[:15]}")
        # Sanity check: kluczowe pliki warsztatowe
        for needed in ("requirements.txt", "notebooks", "data"):
            mark = "✅" if needed in files else "⚠️ BRAK"
            print(f"   {mark} {needed}")
        result_clone = True
    else:
        print(f"❌ BŁĄD git clone (returncode={proc.returncode}):")
        print(proc.stderr[:2000])
except Exception as e:
    print(f"❌ Wyjątek: {type(e).__name__}: {e}")
finally:
    shutil.rmtree(tmp_dir, ignore_errors=True)

## 📋 5. Podsumowanie

Oba ✅ → środowisko firmowe jest gotowe pod warsztat.

In [ ]:
def status(v):
    return "✅ DZIAŁA" if v else "❌ NIE DZIAŁA"

print("WYNIK SMOKE TESTU")
print("=" * 44)
print(f"Projekt:  {PROJECT_ID}")
print(f"Region:   {LOCATION}")
print(f"Model:    {MODEL}")
print("-" * 44)
print(f"Vertex AI (OAuth / Bearer): {status(result_oauth)}")
print(f"Git clone repo:             {status(result_clone)}")
print("=" * 44)

if result_oauth and result_clone:
    print("→ Wszystko gotowe. Można odpalać notebooki warsztatowe na tym środowisku.")
else:
    if not result_oauth:
        print("→ Vertex AI: sprawdź włączone API, rolę aiplatform.user, dostępność modelu")
        print("  w europe-west4 oraz ograniczenia sieciowe (VPC-SC / firewall).")
    if not result_clone:
        print("→ Git clone: sprawdź dostęp sieciowy do GitHub i czy repo jest publiczne.")